# Anchor labels and offsets in NumPy

A detector with anchors is trained on targets. Each anchor is labeled object, background or
ignored by its IoU with the true boxes, and each object anchor gets the four offsets that
would move it onto its true box. Here you compute those targets the way Faster R-CNN does,
for a 3 × 3 grid of 2 × 2 cells, with three anchors centered on each
cell: tall (2 wide, 6 high), square (2 by 2) and wide (6 wide, 2 high). Boxes are
(left, top, right, bottom) in pixels.

**Working through it.** Each part asks for code in the cell marked `# TODO`. Replace the
`raise NotImplementedError` with your code and run the check cell below it. The check prints
`Part N passed`, or stops with a message about what is wrong.

## Setup

Run this cell first.

In [ ]:
import numpy as np

CENTERS = [1, 3, 5]
SHAPES = {"tall": (2, 6), "square": (2, 2), "wide": (6, 2)}     # width, height
ANCHORS, NAMES = [], []
for cy in CENTERS:
    for cx in CENTERS:
        for shape, (w, h) in SHAPES.items():
            half_w, half_h = w / 2, h / 2
            ANCHORS.append([cx - half_w, cy - half_h, cx + half_w, cy + half_h])
            NAMES.append(f"{shape} at ({cx}, {cy})")
ANCHORS = np.array(ANCHORS)
INDEX = dict(zip(NAMES, range(len(NAMES))))      # anchor name to row of ANCHORS
print(ANCHORS.shape, NAMES[12], ANCHORS[12])

## Part 1 (coding task): IoU of every pair

Write `iou_matrix(a, b)` for boxes `a` of shape $(N, 4)$ and `b` of shape $(M, 4)$. It
returns an $(N, M)$ array of IoUs, with `a[i]` against `b[j]` at `[i, j]`. Use
broadcasting, with no Python loops.

In [ ]:
def iou_matrix(a, b):
    """a: (N, 4), b: (M, 4), boxes as (left, top, right, bottom). Returns the (N, M) IoUs."""
    # TODO
    raise NotImplementedError

In [ ]:
truth, A, B = [2, 0, 3, 6], [2, 1, 3, 6], [2, 0, 3.5, 6]
M = iou_matrix(np.array([A, B, truth], dtype=float), np.array([truth, B], dtype=float))
print(M.round(3))
assert M.shape == (3, 2), f"3 boxes against 2 should give a 3 x 2 array, got shape {M.shape}"
assert np.allclose(M, [[5 / 6, 5 / 9], [2 / 3, 1], [1, 2 / 3]]), "the running example's IoUs are 0.833 and 0.667 with the truth, and 0.556 between A and B"
far = iou_matrix(np.array([[0, 0, 1, 1.0]]), np.array([[2, 2, 3, 3.0], [2, 0, 3, 1.0], [1, 0, 2, 1.0]]))
assert np.all(far >= 0), f"an IoU is never negative, got {far.round(3)}"
assert np.allclose(far, 0), f"boxes that do not overlap have IoU 0, got {far.round(3)}: a box below and to the side gives a negative width and a negative height, whose product is positive"
pairs = iou_matrix(ANCHORS, ANCHORS)
assert np.allclose(np.diag(pairs), 1) and np.allclose(pairs, pairs.T), "every anchor has IoU 1 with itself, and IoU is symmetric"
print("Part 1 passed")

## Part 2 (coding task): Labels

Write `label_anchors(anchors, truths, high=0.7, low=0.3)`. It returns two integer arrays
with one entry per anchor:

- `labels`: 1 for an object, 0 for background and $-1$ for an anchor that training
  ignores. An anchor whose best IoU with the true boxes is at least `high` is an object,
  one whose best IoU is below `low` is background, and one in between is ignored.
- `matched`: the index of the true box with which the anchor has its highest IoU.

Then apply Faster R-CNN's second rule: every true box makes its best anchor an object,
whatever that anchor's IoU. When several anchors share a true box's highest IoU, all of
them count as its best.

In [ ]:
def label_anchors(anchors, truths, high=0.7, low=0.3):
    """anchors: (N, 4), truths: (M, 4). Returns labels (N,) in {1, 0, -1} and matched (N,)."""
    # TODO
    raise NotImplementedError

In [ ]:
line = np.array([[2, 0, 3, 6.0]])
labels, matched = label_anchors(ANCHORS, line)
print("objects:", [NAMES[i] for i in np.flatnonzero(labels == 1)])
assert labels.shape == (27,) and matched.shape == (27,), "there should be one label and one match per anchor"
assert labels[INDEX["tall at (3, 3)"]] == 1, "the line's best anchor is tall at (3, 3), IoU 0.5: it is an object even though 0.5 is below 0.7"
assert (labels == 1).sum() == 1 and (labels == 0).sum() == 26, "for the line alone, one object and 26 background anchors"

bar = np.array([[3, 0, 5, 6.0]])
labels, _ = label_anchors(ANCHORS, bar)
tied = [INDEX["tall at (3, 3)"], INDEX["tall at (5, 3)"]]
assert np.all(labels[tied] == 1), "the bar's highest IoU, 1/3, is shared by tall at (3, 3) and tall at (5, 3): both are its best anchors"
assert (labels == 1).sum() == 2, f"the bar should make exactly 2 objects, got {(labels == 1).sum()}"

boxes = np.array([[2, 0, 3, 6.0], [0, 0, 2, 5.0]])
labels, matched = label_anchors(ANCHORS, boxes)
objects = sorted(NAMES[i] for i in np.flatnonzero(labels == 1))
ignored = sorted(NAMES[i] for i in np.flatnonzero(labels == -1))
assert objects == ["tall at (1, 3)", "tall at (3, 3)"], f"objects should be tall at (1, 3) (IoU 0.833) and tall at (3, 3) (the line's best), got {objects}"
assert ignored == ["square at (1, 1)", "square at (1, 3)", "tall at (1, 1)", "tall at (1, 5)"], f"anchors with a best IoU from 0.3 to 0.7 are ignored, got {ignored}"
assert matched[INDEX["tall at (1, 3)"]] == 1 and matched[INDEX["tall at (3, 3)"]] == 0, "matched should hold the index of each anchor's best true box"

labels, _ = label_anchors(ANCHORS, np.array([[4.5, 4.5, 5, 5.0]]))
assert labels[INDEX["square at (5, 5)"]] == 1, "a small box still gets its best anchor, square at (5, 5), with IoU 0.0625"
print("Part 2 passed")

## Part 3 (coding task): Offsets

Write `encode(anchors, boxes)`, which takes anchors and boxes of the same shape $(N, 4)$,
paired row by row, and returns the offsets $(t_x, t_y, t_w, t_h)$ for each
pair. Write `decode(anchors, offsets)`, which turns offsets back into boxes as
(left, top, right, bottom). Both work in center form, a box's center $(x, y)$, width $w$
and height $h$, and the anchor's $(x_a, y_a, w_a, h_a)$:

$$
t_x = (x - x_a) / w_a, \qquad t_y = (y - y_a) / h_a, \qquad
t_w = \log (w / w_a), \qquad t_h = \log (h / h_a),
$$

with natural logarithms.

In [ ]:
def encode(anchors, boxes):
    """anchors, boxes: (N, 4) as (left, top, right, bottom). Returns (N, 4) offsets."""
    # TODO
    raise NotImplementedError


def decode(anchors, offsets):
    """anchors: (N, 4), offsets: (N, 4). Returns (N, 4) boxes as (left, top, right, bottom)."""
    # TODO
    raise NotImplementedError

In [ ]:
line = np.array([[2, 0, 3, 6.0]])
boxes = np.array([[2, 0, 3, 6.0], [0, 0, 2, 5.0]])
tall = ANCHORS[[INDEX["tall at (3, 3)"]]]
t = encode(tall, line)
print(t.round(3))
assert t.shape == (1, 4), f"one anchor and one box give one row of 4 offsets, got shape {t.shape}"
assert np.allclose(t, [[-0.25, 0, np.log(0.5), 0]]), f"the running example's offsets are -0.25, 0, -0.693, 0, got {t.round(3)}"
wide = ANCHORS[[INDEX["wide at (3, 1)"]]]
t = encode(wide, np.array([[0, 1, 6, 2.0]]))
assert np.allclose(t, [[0, 0.25, 0, np.log(0.5)]]), f"for wide at (3, 1) and the box (0, 1, 6, 2) the offsets are 0, 0.25, 0, -0.693, got {t.round(3)}"
box = decode(tall, np.array([[0.5, 0, np.log(2), 0]]))
assert np.allclose(box, [[2, 0, 6, 6]]), f"moving tall at (3, 3) by half its width and doubling the width gives (2, 0, 6, 6), got {box}"

rng = np.random.default_rng(3)
corner = rng.uniform(0, 4, size=(27, 2))
random_boxes = np.hstack([corner, corner + rng.uniform(0.5, 3, size=(27, 2))])
assert np.allclose(decode(ANCHORS, encode(ANCHORS, random_boxes)), random_boxes), "decode should undo encode"

labels, matched = label_anchors(ANCHORS, boxes)
objects = np.flatnonzero(labels == 1)
targets = encode(ANCHORS[objects], boxes[matched[objects]])
print([NAMES[i] for i in objects], targets.round(3))
assert np.allclose(targets, [[0, -0.5 / 6, 0, np.log(5 / 6)], [-0.25, 0, np.log(0.5), 0]]), "the two object anchors' targets are wrong"
print("Part 3 passed")

## Solution

### Part 1: IoU of every pair

In [ ]:
def iou_matrix(a, b):
    """a: (N, 4), b: (M, 4), boxes as (left, top, right, bottom). Returns the (N, M) IoUs."""
    left_top = np.maximum(a[:, None, :2], b[None, :, :2])        # (N, M, 2)
    right_bottom = np.minimum(a[:, None, 2:], b[None, :, 2:])
    sides = np.clip(right_bottom - left_top, 0, None)             # no overlap: 0, not negative
    overlap = sides[..., 0] * sides[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return overlap / (area_a[:, None] + area_b[None, :] - overlap)

**Why it works.** Indexing with `None` turns `a` into shape $(N, 1, 4)$ and `b` into $(1, M, 4)$, so
the maximum of the left and top edges and the minimum of the right and bottom edges are
taken for every pair at once. When the boxes do not overlap, right minus left or bottom
minus top is negative, and clipping each side at 0 before multiplying is what makes the
overlap 0. Clipping only the product fails for a box that is both below and to the side,
whose two negative sides multiply to a positive area.

### Part 2: Labels

In [ ]:
def label_anchors(anchors, truths, high=0.7, low=0.3):
    """anchors: (N, 4), truths: (M, 4). Returns labels (N,) in {1, 0, -1} and matched (N,)."""
    iou = iou_matrix(anchors, truths)                  # (N, M)
    best = iou.max(axis=1)
    labels = np.where(best >= high, 1, np.where(best < low, 0, -1))
    top = iou.max(axis=0)                              # each true box's highest IoU
    labels[(iou == top).any(axis=1)] = 1               # its best anchors, ties included
    return labels, iou.argmax(axis=1)

**Why it works.** The thresholds look at each anchor's best IoU over the true boxes, the maximum
along axis 1. The second rule looks the other way, at each true box's best IoU over the
anchors, the maximum along axis 0. Comparing the whole IoU matrix with that row of maxima
marks every anchor that reaches a true box's best, so a tie marks both anchors, where
`argmax` would pick only the first. The rule is applied last, so it overrides the
thresholds: the running example's tall anchor, at 0.5, would otherwise be ignored and the
line would have no anchor to learn from.

### Part 3: Offsets

In [ ]:
def center_form(boxes):
    """(left, top, right, bottom) to center x, center y, width, height."""
    w = boxes[:, 2] - boxes[:, 0]
    h = boxes[:, 3] - boxes[:, 1]
    return boxes[:, 0] + w / 2, boxes[:, 1] + h / 2, w, h


def encode(anchors, boxes):
    """anchors, boxes: (N, 4) as (left, top, right, bottom). Returns (N, 4) offsets."""
    xa, ya, wa, ha = center_form(anchors)
    x, y, w, h = center_form(boxes)
    return np.stack([(x - xa) / wa, (y - ya) / ha, np.log(w / wa), np.log(h / ha)], axis=1)


def decode(anchors, offsets):
    """anchors: (N, 4), offsets: (N, 4). Returns (N, 4) boxes as (left, top, right, bottom)."""
    xa, ya, wa, ha = center_form(anchors)
    x = xa + offsets[:, 0] * wa
    y = ya + offsets[:, 1] * ha
    w = wa * np.exp(offsets[:, 2])
    h = ha * np.exp(offsets[:, 3])
    return np.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], axis=1)

**Why it works.** Both functions go through center form, because the offsets measure the move of
the center in units of the anchor's own size and the change of size as a log ratio. `x`
divides by the anchor's width and `y` by its height, which matters for anchors that are
not square: the wide anchor's $t_y$ is $0.5 / 2$, not $0.5 / 6$. Decoding runs each step
backwards, multiplying by the anchor's size and taking `exp` of the log ratios, then goes
back to corners. In the last check, the box (0, 0, 2, 5) has center (1, 2.5) and height
5, and its anchor, tall at (1, 3), has center (1, 3) and height 6, so its targets are
$0$, $-0.5 / 6$, $0$ and $\log(5/6)$.